# `ammico` workshop November 2026: comparing models with OpenRouter

*Delia Dumitrescu, Inga Ulusoy, Dmitrii Kapitan*

With `ammico`, you can analyze text on images and image content at the same time, see https://ssciwr.github.io/AMMICO/. In this notebook, we run the same analysis with several vision-language models hosted on [OpenRouter](https://openrouter.ai) and compare the results. First we start by importing the packages.

In [ ]:
import openrouter_utils as OR

import ammico

This imports all the functionality from `ammico`. To analyze images, you can manually upload images to the Jupyter server; in this workshop, we will use pre-prepared datasets in `/home/jovyan/data_mounts/webdav`. The helper functions for running several models are in `openrouter_utils.py`, next to this notebook.

# Step 1: Read your data into AMMICO

`ammico` reads in files from a directory. You can iterate through directories in a recursive manner and filter by extensions. Note that the order of the files may vary on different OS. Reading in these files creates a dictionary `image_dict`, with one entry per image file, containing the file path and filename. This dictionary is the main data structure that ammico operates on and is extended successively with each detector run as explained below.

For reading in the files, the ammico function `find_files` is used, with optional keywords:

| input key | input type | possible input values |
| --------- | ---------- | --------------------- |
`path` | `str` | the directory containing the image files (defaults to the location set by environment variable `AMMICO_DATA_HOME`) |
| `pattern` | `str\|list` | the file extensions to consider (defaults to "png", "jpg", "jpeg", "gif", "webp", "avif", "tiff") |
| `recursive` | `bool` | include subdirectories recursively (defaults to `True`) |
| `limit` | `int` | maximum number of files to read (defaults to `20`, for all images set to `None` or `-1`) |
| `random_seed` | `int` | the random seed for shuffling the images; applies when only a few images are read and the selection should be preserved (defaults to `None`) |

Since every model is billed per request on OpenRouter, you may want to start with a small number of images by setting `limit`. All models are run on exactly the same images.

In [ ]:
# Define your data path
data_path = (
    "/home/jovyan/data_mounts/webdav/dataset_politicians/"  # the selected directory
)

# Find files and create the image dictionary
image_dict = ammico.find_files(
    path=data_path,
    limit=-1,  # Limit the number of files to process (optional)
)

# 2. Run the content analysis: Image summary

As an example we will create an image caption ("Summary") with several vision-language models and compare them.

[OpenRouter](https://openrouter.ai) gives access to many hosted models (OpenAI, Google, Anthropic, Qwen, Meta, Mistral, ...) through a single **OpenAI-compatible API** and a single API key. You need an account and an API key, which you can create at [openrouter.ai/keys](https://openrouter.ai/keys). The key is read from the environment variable `OPENROUTER_API_KEY`; if it is not set, you are asked to enter it below. Do not write your API key directly into the notebook, in particular if you share the notebook with others!

**Privacy note:** with OpenRouter your images are sent to a third party (OpenRouter and the provider of the selected model). Do not use this setup for data that must not leave your institution.

Any model on OpenRouter that accepts images as input can be used, see the [OpenRouter model list](https://openrouter.ai/models?input_modalities=image), where you also see the price per token. The model id is the name shown on the model page, prefixed with the provider, for example `openai/gpt-4o-mini`. Add or remove model ids in the list below.

In [ ]:
# Connect to OpenRouter (asks for the API key if OPENROUTER_API_KEY is not set)
OR.connect()

models = [
    "google/gemini-2.5-flash",
    "qwen/qwen2.5-vl-72b-instruct",
    "meta-llama/llama-4-maverick",
]

Then, we create an instance of the Python class that handles the image summary and visual question answering tasks. It holds one `ammico.ImageSummaryDetector` per model, each working on its own copy of `image_dict`, so that the results of the different models do not overwrite each other.

Many models on OpenRouter are served from a pool shared between all users. When it is busy, the request is rejected with a **rate limit error** (`Error code: 429`); such requests are retried with increasing waiting times. To avoid the rate limit, the number of requests sent at the same time is limited by `max_parallel_images` and `max_parallel_requests` (one request per question); lower these numbers if you still see many rate limit errors. Reasoning models "think" before they answer, which can leave the caption empty, so reasoning is switched off (`disable_reasoning=True`).

In [ ]:
image_summary_vqa = OR.MultiModelDetector(
    models,
    image_dict,
    max_parallel_images=4,  # images processed at the same time
    max_parallel_requests=2,  # requests (questions) per image at the same time
    disable_reasoning=True,
)

After this, we can create the image captions. Depending on the number of images and models, this can take several minutes. If a request still fails after all retries, a warning is printed and the image is left without a caption. When you execute the cell again, only the images without results are sent to the models again.

In [ ]:
summary = image_summary_vqa.analyse_images_from_dict(
    analysis_type="summary", is_concise_summary=True
)

The results are provided in a dictionary with one updated image dictionary per model. Execute the cell below to see the image that was analyzed together with the captions (summaries) of all models.

In [ ]:
OR.show_summaries(summary)

# 3. Run the content analysis: Visual question answering

You may also ask questions about the images. For this, provide a list of questions and pass it to the Python class that you have instantiated above. All models answer the same questions:

In [ ]:
list_of_questions = [
    "Is there a politician in the picture?",
    "Is there a politician in the picture? Answer yes, no or dk. Don't guess.",
    "Is there a politician in the picture? Answer yes, no, or dk. Briefly justify your answer and, if there is a politician, give the name. Don't guess.",
    "Is any of the following politicians in the picture: Donald Trump, Volodymyr Zelenskyy or Vladimir Putin? Answer yes or no. If yes, give the name. Don't guess.",
]  # add or replace with your own questions

In [ ]:
answers = image_summary_vqa.analyse_images_from_dict(
    analysis_type="questions",
    list_of_questions=list_of_questions,
    is_concise_summary=True,
    is_concise_answer=True,
)

Again for your convenience we display the images and the answers of all models to the questions together below:

In [ ]:
OR.show_answers(answers, list_of_questions)

# 4. Export the results

To export the results for further processing, convert the image dictionaries of all models into one pandas dataframe. Each row corresponds to one image analyzed by one model; the column `model` tells you which model produced the result.

In [ ]:
image_df = OR.get_dataframe(answers)

Inspect the dataframe:

In [ ]:
image_df.head(5)

Export the dataframe to a csv file, and additionally the results of each model to its own csv file (`<provider>__<model>.csv`):

In [ ]:
image_df.to_csv("./politicians_data_out.csv")
OR.save_per_model(image_df, "./politicians_results")

# 5. Analyse the results
You can compare the results to the manual labels and observe how accurate the different VLMs capture the content.

# Question accuracy for several models

Which of several questions (prompts) gets the most images right for each model, and do the models agree on which question works best?

**How to use:** run the VQA with ammico on one of the course datasets and save the result as a csv file. Fill in the two paths in the next cell and run all cells. Tables and plots are saved in `comparison_results/` next to the (first) result file.

In [ ]:
# ============ SETTINGS: the only cell you need to change ============
RESULTS_FILES = [  # csv file(s) with the model answers, one file per model
    "/home/jovyan/notebooks/politicians_results/google__gemini-2.5-flash.csv",  # (or one file with several models); glob patterns work,
    "/home/jovyan/notebooks/politicians_results/meta-llama__llama-4-maverick.csv",  # e.g. "../my_results/emotions_*.csv"
    "/home/jovyan/notebooks/politicians_results/qwen__qwen2.5-vl-72b-instruct.csv",  # e.g. "../my_results/emotions_*.csv"
]
IMAGE_DIR = data_path  # folder with the images, here the same as above

**Datasets** – the dataset (emotions, politicians or symbols) is detected from the image names in the result file, so only the paths change. The paths are relative to this notebook.

The images can be in one flat folder; they are only needed for the image examples. Questions, ground truth and expected answers are defined in `DATASETS` in `vqa_eval_utils.py`; the ground-truth and question files are read from `ground_truth/` next to the image folder (or from the `*_output/` folders of the course repository). The answers must be to the questions in these files, in the same order.

## 5.1 Data and questions

In [ ]:
import vqa_eval_utils as U

run = U.load(RESULTS_FILES, IMAGE_DIR, "question_accuracy_models")
U.show_questions(run)

## 5.2 From free text to answers
Each answer is mapped to a category (yes / no / dk, …) by the rules of its question type. `exact`, `first` and `package_code` are safe; the keyword rules (`kw_*`) are heuristics, so read the sample of raw answers below against their `normalised` value. Only images answered by all models are used, so all models are compared on the same images.

In [ ]:
U.show_normalisation(run)

## 5.3 Accuracy per question
**Accuracy** = share of images whose (normalised) answer matches the expected answer from the ground truth. DK, unclear and unparsed answers count as **wrong**; images without an expected answer (e.g. label `unclear`) are not scored.
The **majority baseline** is the accuracy of always giving the most frequent expected answer – a question is only useful if it beats it clearly. Error bars are 95% Wilson confidence intervals.

In [ ]:
acc = U.show_accuracy(run)

## 5.4 Accuracy per image group
The same accuracy for each image group and model: a low value shows which kind of image makes a question fail.

In [ ]:
U.show_groups(run, acc)

## 5.5 Which differences between questions are significant?
Each pair of questions is compared on the same images with an exact McNemar test (Holm-corrected). Blue = the row question is more accurate, red = less accurate, * = significant.

In [ ]:
tests = U.show_pairs(run)

## 5.6 Summary: which question to use?
The most accurate question per model and the questions that are not significantly worse. A question that is never significantly worse is a safe choice across models.

In [ ]:
U.show_best(run, acc, tests)

## 5.7 Difficult images
How many answers go wrong per image, and the hardest images of each group. Errors on every question point to a hard image or a questionable label; errors on some questions point to the wording.

In [ ]:
U.show_difficult_images(run)